#Data Reading

## Common dbutils Modules

### 1. dbutils.fs (File System Utilities)
Used to work with DBFS (Databricks File System).

**List files:**
```python
dbutils.fs.ls("/FileStore/")
```

**Create directory:**
```python
dbutils.fs.mkdirs("/FileStore/test")
```

**Copy files:**
```python
dbutils.fs.cp(
    "dbfs:/FileStore/source.csv",
    "dbfs:/FileStore/target.csv"
)
```

**Remove files:**
```python
dbutils.fs.rm("/FileStore/test", recurse=True)
```

---

### 2. dbutils.secrets (Secret Management)
Used to securely access passwords, API keys, and tokens.

**List secret scopes:**
```python
dbutils.secrets.listScopes()
```

**Get secret value:**
```python
dbutils.secrets.get(scope="my-scope", key="api-key")
```


In [0]:
dbutils.fs.ls('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/')

##Data Reading CSV

In [0]:
df_csv = spark.read.format('csv').option('inferschema', True).option('header', True).load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')

In [0]:
df_csv.show()
df_csv.display()

##Data Reading JSON

In [0]:
df_json = spark.read.format('json').option('inferschema', True)\
            .option('header',True)\
            .option('multiline', False)\
            .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/drivers.json')

In [0]:
df_json.show()
df_json.display()

#Schema Definition

In [0]:
df_csv.printSchema()
df_json.printSchema()



We can define schema to the table or data Frame in pyspark using the DDL Schmema or the Struct Type Schema




##DDL SCHEMA

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df_csv = spark.read.format('csv')\
                    .option('inferschema', True)\
                    .option('header', True)\
                    .schema(my_ddl_schema)\
                    .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')

In [0]:
df_csv.printSchema()

##StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *  

In [0]:
my_struct_csv_schema  = StructType([ StructField('Item_Identifier',StringType(),True), StructField('Item_Weight',StringType(),True), StructField('Item_Fat_Content',StringType(),True), StructField('Item_Visibility',StringType(),True), StructField('Item_MRP',StringType(),True), StructField('Outlet_Identifier',StringType(),True), StructField('Outlet_Establishment_Year',StringType(),True), StructField('Outlet_Size',StringType(),True), StructField('Outlet_Location_Type',StringType(),True), StructField('Outlet_Type',StringType(),True), StructField('Item_Outlet_Sales',StringType(),True)
])


In [0]:
df_my_struct_csv = spark.read.format('csv')\
                        .option('infershema', True)\
                        .option('header', True)\
                        .schema(my_struct_csv_schema)\
                        .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')


In [0]:
df_my_struct_csv.printSchema()

#TRANSFORMATIONS

In [0]:
df_csv.printSchema()

##SELECT

In [0]:
df_csv.printSchema()

## Select Column Methods

**Two ways to select columns in PySpark:**

1. **String-based selection** - `select("col1", "col2")`
   * Uses string column names
   * Suitable for simple column selection
   * Quick and concise

2. **Column object-based selection** - `select(col("col1"), col("col2"))`
   * Uses PySpark Column objects
   * Allows additional operations:
     * Aliasing (`.alias()`)
     * Expressions and calculations
     * Casting (`.cast()`)
     * Transformations

**Note:** For simple selection, both produce the same result.

**Example with Column objects:**
```python
from pyspark.sql.functions import col

df_csv.select(
    col("Item_Identifier").alias("ID"),
    col("Item_Weight") * 2
).show()
```


In [0]:
df_csv.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

In [0]:
df_csv.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

##ALIAS

In [0]:
df_csv.select(col('Item_Identifier').alias('Item_ID')).display()

###FILTER

In [0]:
df_csv.display()

![image_1787641808078.png](./image_1787641808078.png "image_1787641808078.png")

###Scenario - 1


In [0]:
df_csv.filter((col('Item_Fat_Content') == 'Regular')).display()

###Scenario - 2

In [0]:
df_csv.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') <10 ) ).display()

###Scenarion - 3

# PySpark Quick Reference: `isNull()` and `isin()`

## 🔍 Null Checking

**Correct way to check for NULL values:**

```python
col("column_name").isNull()      # Finds null values
col("column_name").isNotNull()   # Finds non-null values
```

**❌ Common mistakes (do NOT use):**

```python
col("column_name") == NULL  # Wrong!
col("column_name") == None  # Wrong!
```

> **Why?** Normal equality comparison (`==`) cannot correctly identify SQL NULL values.

---

## 📊 Value Matching

### Checking a Single Value

```python
col("city") == "Chennai"
```

### Checking Multiple Possible Values

**Using `isin()` (recommended):**

```python
col("city").isin("Chennai", "Bangalore")
```

**This is equivalent to:**

```python
(col("city") == "Chennai") | (col("city") == "Bangalore")
```

> **Key Point:** `isin()` uses **OR** logic because a column can match **any one** of the provided values.

---

## ⚠️ Common Mistake: Using AND Instead of OR

**❌ This is INCORRECT:**

```python
(col("city") == "Chennai") & (col("city") == "Bangalore")
```

> **Why it fails:** This returns **no rows** because one column value cannot be both "Chennai" **and** "Bangalore" at the same time.

---

## 🚫 Excluding Multiple Values

```python
~col("city").isin("Chennai", "Bangalore")
```

**Equivalent to:**

```python
(col("city") != "Chennai") & (col("city") != "Bangalore")
```

---

## 🔧 Logical Operators

| Operator | Meaning | Example |
|----------|---------|----------|
| `&` | AND | `(condition1) & (condition2)` |
| `\|` | OR | `(condition1) \| (condition2)` |
| `~` | NOT | `~condition` |

> **Important:** Always put each PySpark condition inside **parentheses**.

---

## 💡 Easy Rules to Remember

| Function | Use Case |
|----------|----------|
| `isNull()` | Check for NULL values |
| `==` | Check for one specific value |
| `isin()` | Check if value matches **any** from multiple options |

---

## 🎯 Key Takeaway

**`isin("value1", "value2")` is equivalent to equality conditions joined using `\|` (OR), not `&` (AND).**

In [0]:
df_csv.filter((col('Outlet_Location_Type').isin('Tier 1', 'Tier 2')) & (col('Outlet_Size').isNull())).display()

## `withColumnRenamed()`

**Purpose:** Renames a single column in a DataFrame.

**Syntax:** `df.withColumnRenamed("old_name", "new_name")`

**Returns:** A new DataFrame with the renamed column (original DataFrame remains unchanged).

**Example:**
```python
df_csv.withColumnRenamed("Item_Identifier", "Product_ID")
```

**Note:** To rename multiple columns, chain multiple `withColumnRenamed()` calls or use `toDF()` with a list of new names.

In [0]:
df_csv.withColumnRenamed('Item_Weight', 'Item_Wt').display()

## `withColumn()`

**Purpose:** Adds a new column or replaces an existing column in a DataFrame.

**Syntax:** `df.withColumn("column_name", expression)`

**Returns:** A new DataFrame with the added/modified column (original DataFrame remains unchanged).

**Common use cases:**
* Add a new calculated column
* Transform an existing column
* Cast column to a different data type
* Add constant values

**Examples:**
```python
# Add a new column with calculation
df.withColumn("Discounted_Price", col("Item_MRP") * 0.9)

# Modify existing column (cast to different type)
df.withColumn("Item_Weight", col("Item_Weight").cast("double"))

# Add a constant column
df.withColumn("Country", lit("India"))
```

###Scenario - 1

###lit() -- The lit function in PySpark is used to create a new column with a constant or literal value in a DataFrame.

In [0]:
df_csv.withColumn('Flag', lit('New')).display()

In [0]:
df_csv.withColumn('Multiply', (col('Item_Weight')*col('Item_MRP'))).display()

###Scenario - 2

## `regexp_replace()`

**Purpose:** Replaces all substrings that match a regular expression pattern with a replacement string.

**Syntax:** `regexp_replace(column, pattern, replacement)`

**Parameters:**
* `column` - The column to search in
* `pattern` - Regular expression pattern to match
* `replacement` - String to replace matched patterns

**Common use cases:**
* Clean and standardize text data
* Remove unwanted characters
* Replace multiple variations with a single value
* Format strings (e.g., phone numbers, IDs)

**Examples:**
```python
# Remove special characters
df.withColumn("Clean_Name", regexp_replace(col("Name"), "[^a-zA-Z0-9 ]", ""))

# Standardize text variations
df.withColumn("Fat_Content", regexp_replace(col("Item_Fat_Content"), "low fat|LF", "Low Fat"))

# Format phone numbers
df.withColumn("Phone", regexp_replace(col("Phone"), "(\\d{3})(\\d{3})(\\d{4})", "($1) $2-$3"))
```

In [0]:
df = df_csv.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),'Regular', 'Reg'))\
      .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),'Low Fat', 'Lf'))

df.display()

###Type Casting

**Purpose:** Converts a column from one data type to another using the `cast()` method.

**Syntax:** `col("column_name").cast("target_type")` or `col("column_name").cast(TargetType())`

**Common use cases:**
* Convert string to numeric for calculations
* Convert numeric to string for formatting
* Change precision (e.g., `IntegerType` to `DoubleType`)

**Example:**
```python
df.withColumn("Item_Weight", col("Item_Weight").cast("double"))
df.withColumn("Item_Weight", col("Item_Weight").cast(StringType()))
```

In [0]:
df_csv.withColumn('Item_Weight', col('Item_Weight').cast(StringType())).display()

In [0]:
df_csv.select(col("Item_Weight").cast(StringType())).display()

###Sorting

**Purpose:** Arranges DataFrame rows in ascending or descending order based on one or more columns.

In Pyspark sort() and orderBy() are functional aliases of each other, meaning they share the exact same underlying implementation and can be used interchangeably to arrange your DataFrame rows.

**Functions:**
* `orderBy()` – Sorts rows (same as `sort()`)
* `sort()` – Alias of `orderBy()`
* `asc()` – Ascending order (default)
* `desc()` – Descending order

**Syntax:**
```python
df.orderBy(col("column_name").asc())
df.orderBy(col("column_name").desc())
df.orderBy(col("col1").desc(), col("col2").asc())
```

**Common use cases:**
* Sort products by price (high to low)
* Sort by multiple columns (primary + secondary)
* Rank items by sales or weight

###Scenarion - 1: Basic Sorting (Single Column)

In [0]:
#Using sort 
#Ascending is default no need to mention .asc()
df_csv.sort(col("Item_Weight").desc()).display()

#using orderby

df_csv.orderBy(col("Item_Weight").desc()).display()

###Scenarion - 2: Sorting with Ascending Parameter

In [0]:
#There should be no space between the ascending and =
df_csv.sort(col("Item_Visibility"), ascending=False).display()

df_csv.sort(col("Item_Visibility"), ascending=1).display()

#There should be no space between the ascending and =
df_csv.orderBy(col("Item_Visibility"), ascending=False).display()

df_csv.orderBy(col("Item_Visibility"), ascending=1).display()


###Scenarion - 3: Sorting for multiple columns

In [0]:
df_csv.sort(["Item_Weight", "Item_Visibility"], ascending=[0,1]).display()

df_csv.orderBy(["Item_Weight", "Item_Visibility"], ascending=[1,0]).display()
df_csv.orderBy(col('Item_Weight').asc(),col('Item_Visibility').desc() ).display()

In [0]:
df_csv.select(col('Item_Identifier'), col("Item_Fat_Content"), col("Item_Weight"), col("Item_Visibility")).filter(col("Item_Fat_Content") == "Low Fat" ).orderBy(["Item_Weight", "Item_Visibility"], ascending=[1,0]).display()

###Limit

In [0]:
df_csv.limit(5).display()

###Drop

###Scenario - 1

In [0]:
df_csv.drop("Item_Fat_Content").display()

###Scenario - 2

In [0]:
df_csv.drop("Item_Fat_Content", "Item_Visibility").display()

###Drop Duplicates

**Purpose:** Removes duplicate rows from a DataFrame, keeping only unique records.

**Functions:**
* `dropDuplicates()` – Removes duplicate rows (same as `drop_duplicates()`)
* `drop_duplicates()` – Alias of `dropDuplicates()`

**Syntax:**
```python
# Drop duplicates across all columns
df.dropDuplicates()
```
When you call dropDuplicates() with no arguments, it only removes rows where every single column has the same value — i.e., a complete duplicate row. It does not remove a row just because two columns happen to share the same value.

```python
# Drop duplicates based on specific columns
df.dropDuplicates(["col1", "col2"])
```
it will remove the entire row. When you specify columns in dropDuplicates(["col1", "col2"]), PySpark only looks at those two columns to decide what counts as a duplicate. If two rows have identical values in those two columns, it keeps only one row and drops the other — even if all other columns have different values.

**Common use cases:**
* Remove fully duplicated rows
* Deduplicate by key columns (e.g., keep one row per `Item_Identifier`)
* Clean data after joins or unions that produce duplicates

In [0]:
df_csv.dropDuplicates().display()

#you can also use distinct(same way used in sql)

df_csv.distinct().display()

In [0]:
df_csv.dropDuplicates(["Item_Type", "Item_Fat_Content"]).display()

df_csv.dropDuplicates(subset = ["Item_Type"]).display()